# VocalCoach — Training Plan v2 (post-overnight analysis)

Runs derived from overnight analysis (Runs 52–57). Key findings that drive this plan:

| Finding | Impact |
|---|---|
| **attn4 backbone (Run 57)** best Stage-1 ever: OOD VDR 46.9%, RPA 99.1%, VDR 82.1% | Use as base for all new runs |
| GTSinger tech labels hurt probe-mode mF1 (52: 0.569 vs 48: 0.657) | Probe runs: VocalSet+AnnotatedVocalSet only |
| difflr joint (Run 53) best OOD-tested joint: mF1 0.676, OOD VDR 30.3% | Re-run from attn4 backbone |
| warmjoint→deepprobe (Run 55): OOD VDR 44.0%, new best for technique-trained ckpt | Re-run from attn4 backbone |
| Note head collapses OOD VDR even with deep head (Run 56: 23.0%) | Must be trained in Stage-1 alongside pitch+VAD |
| Quality runs (Q1/Q2) need evalQuality.py for scoring | New eval script + cells added |

**Run order:**

| ID | Run | Type | Base | New? |
|---|---|---|---|---|
| R1 | `stage2_attn4_deepprobe` | Probe, deep head, VS+AVS only | attn4 backbone | ✓ |
| R2 | `stage2_attn4_warmjoint` | Warm joint 20ep, lr=1e-4 | attn4 backbone | ✓ |
| R3 | `stage2_attn4_warmjoint_deepprobe` | Deep probe from R2 | R2 | ✓ |
| R4 | `stage2_attn4_difflr` | Joint difflr, deep head | attn4 backbone | ✓ |
| Q1 | `stage2_quality_v3_attn4` | Quality V3 distill | R1 | ✓ |
| Q2 | `stage2_quality_v2_attn4` | Quality V2 9-dim ccmusic | R1 | ✓ |
| S1N | `stage1_attn4_notehead` | Stage-1 + note head from scratch | attn4 backbone | ✓ |
| EVAL | Quality eval (R1 Q1/Q2 checkpoints) | evalQuality.py | — | ✓ |

R1 and R4 can run in parallel. R2→R3 must be sequential. Q1/Q2 run after R1.

---

### Checkpoints to upload

Two checkpoints needed — upload via Cell 1b:
1. `stage1_tcn_256_conformerattn2_lowlr/best_metric.pth` — Run 45 (already on Drive)
2. `stage1_tcn_256_conformerattn4_lowlr/best_metric.pth` — Run 57 (new, upload from local)

### Data zip (unchanged from previous plan)
```bash
cd ~/NanoPitch-MusicalAI
zip -1 NanoPitch_data_plan.zip \
    data/clean.npz data/noise.npz data/test.npz \
    data/merged_pitchvad/clean.npz \
    data/vocalset/technique_train.npz data/vocalset/technique_test.npz \
    data/annotated_vocalset/note_train.npz data/annotated_vocalset/note_test.npz \
    data/gtsinger_technique/technique_train.npz \
    data/gtsinger_technique/technique_gtsinger_train.npz \
    data/gtsinger_technique/technique_gtsinger_test.npz \
    data/quality/quality_mse.npz data/quality/quality_ccmusic.npz \
    data/quality_50k/quality_pairs.npz
```

## Cell 1 — GPU check + batch size

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU:            {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM:           {vram_gb:.1f} GB')

BATCH_SIZE   = 128 if vram_gb > 30 else 64 if vram_gb > 15 else 32
NUM_WORKERS  = 8
print(f'\nUsing batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}')

## Cell 1b — Upload backbone checkpoint (if not already on Drive)

Run this cell **once** to upload `best_metric.pth` for `stage1_tcn_256_conformerattn2_lowlr`  
from your local machine. Skip if the file is already in Drive.

In [ ]:
import os
from google.colab import files

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'

# Upload Run 57 attn4 backbone (new — not yet on Drive)
ATTN4_DEST = f'{RUNS_DIR}/stage1_tcn_256_conformerattn4_lowlr/checkpoints'
attn4_path = f'{ATTN4_DEST}/best_metric.pth'

if os.path.exists(attn4_path):
    print(f'attn4 checkpoint already on Drive ({os.path.getsize(attn4_path)/1e6:.0f} MB) — skip.')
else:
    print('Upload best_metric.pth for stage1_tcn_256_conformerattn4_lowlr (Run 57):')
    print('  Local path: vocalcoach/runs/stage1_tcn_256_conformerattn4_lowlr/checkpoints/best_metric.pth')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        os.makedirs(ATTN4_DEST, exist_ok=True)
        dest = f'{ATTN4_DEST}/best_metric.pth'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'Saved → {dest}  ({len(data)/1e6:.1f} MB)')

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, numpy as np

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'
os.makedirs(RUNS_DIR, exist_ok=True)

# All files extracted from NanoPitch_data_plan.zip → /content/
# Two clean.npz files serve different purposes:
#   /content/data/clean.npz              = GTSinger only (40,940 clips) — eval for probe runs
#   /content/data/merged_pitchvad/clean.npz = GTSinger+VocalSet (51,627 clips) — joint training (P1, P3)
files_needed = {
    'data/clean.npz':                                          '/content/data/clean.npz',
    'data/noise.npz':                                          '/content/data/noise.npz',
    'data/test.npz':                                           '/content/data/test.npz',
    'data/merged_pitchvad/clean.npz':                          '/content/data/merged_pitchvad/clean.npz',
    'data/vocalset/technique_train.npz':                       '/content/data/vocalset/technique_train.npz',
    'data/vocalset/technique_test.npz':                        '/content/data/vocalset/technique_test.npz',
    'data/annotated_vocalset/note_train.npz':                  '/content/data/annotated_vocalset/note_train.npz',
    'data/annotated_vocalset/note_test.npz':                   '/content/data/annotated_vocalset/note_test.npz',
    'data/gtsinger_technique/technique_train.npz':             '/content/data/gtsinger_technique/technique_train.npz',
    'data/gtsinger_technique/technique_gtsinger_train.npz':    '/content/data/gtsinger_technique/technique_gtsinger_train.npz',
    'data/gtsinger_technique/technique_gtsinger_test.npz':     '/content/data/gtsinger_technique/technique_gtsinger_test.npz',
    'data/quality/quality_mse.npz':                            '/content/data/quality/quality_mse.npz',
    'data/quality/quality_ccmusic.npz':                        '/content/data/quality/quality_ccmusic.npz',
    'data/quality_50k/quality_pairs.npz':                      '/content/data/quality_50k/quality_pairs.npz',
}

def _npz_valid(path):
    if not os.path.exists(path): return False
    try:
        with zipfile.ZipFile(path, 'r'): return True
    except zipfile.BadZipFile:
        return False

missing = []
for name, dest in files_needed.items():
    if not _npz_valid(dest):
        if os.path.exists(dest):
            print(f'  CORRUPTED (re-extracting): {dest}')
            os.remove(dest)
        missing.append(name)

if missing:
    zip_path = f'{DRIVE_ROOT}/NanoPitch_data_plan.zip'
    print(f'Extracting {len(missing)} file(s) from NanoPitch_data_plan.zip ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        for name in missing:
            dest = files_needed[name]
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            z.extract(name, '/content/')
            if _npz_valid(dest):
                print(f'  OK  {name}')
            else:
                raise RuntimeError(f'Extraction failed: {dest}')
else:
    print('All data files present.')

!ls -lh /content/data/
!ls -lh /content/data/merged_pitchvad/
!ls -lh /content/data/quality/ 2>/dev/null
!ls -lh /content/data/quality_50k/ 2>/dev/null

## Cell 3 — Clone repo and install dependencies

In [ ]:
import os

REPO_DIR = '/content/NanoPitch-MusicalAI'
REPO_URL = 'https://github.com/rajat17-personal/NanoPitch-MusicalAI'
BRANCH   = 'feat/finalProject'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('Repo present — pulling latest...')
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    print('Cloning...')
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -r requirements.txt --quiet
print('Setup complete.')

## Cell 4 — Verify data and restore backbone checkpoint

In [ ]:
import numpy as np, os

def check(label, path, key='lengths'):
    try:
        d = np.load(path)
        n = d[key].shape[0]
        print(f'  OK  {label:<55} {n:>6} clips')
    except Exception as e:
        print(f'  MISSING  {label:<52} {e}')

print('=== Pitch/VAD data ===')
check('clean.npz',                                '/content/data/clean.npz')
check('noise.npz',                                '/content/data/noise.npz')
check('test.npz',                                 '/content/data/test.npz', key='clips')

print('\n=== Technique data ===')
check('vocalset/technique_train.npz',             '/content/data/vocalset/technique_train.npz')
check('vocalset/technique_test.npz',              '/content/data/vocalset/technique_test.npz')
check('gtsinger_technique/technique_train.npz',   '/content/data/gtsinger_technique/technique_train.npz')
check('gtsinger_technique/gtsinger_train.npz',    '/content/data/gtsinger_technique/technique_gtsinger_train.npz')

print('\n=== Note head data ===')
check('annotated_vocalset/note_train.npz',        '/content/data/annotated_vocalset/note_train.npz')
check('annotated_vocalset/note_test.npz',         '/content/data/annotated_vocalset/note_test.npz')

print('\n=== Quality head data ===')
check('quality/quality_mse.npz',                  '/content/data/quality/quality_mse.npz', key='lengths')
check('quality/quality_ccmusic.npz',              '/content/data/quality/quality_ccmusic.npz', key='lengths')
# quality_pairs uses lengths_pro not lengths
try:
    d = np.load('/content/data/quality_50k/quality_pairs.npz')
    n = d['lengths_pro'].shape[0]
    print(f'  OK  quality_50k/quality_pairs.npz{"":>22} {n:>6} pairs')
except Exception as e:
    print(f'  MISSING  quality_50k/quality_pairs.npz  {e}')

print('\n=== Backbone checkpoint ===')
BACKBONE_CKPT = f'{RUNS_DIR}/stage1_tcn_256_conformerattn2_lowlr/checkpoints/best_metric.pth'
if os.path.exists(BACKBONE_CKPT):
    sz = os.path.getsize(BACKBONE_CKPT)/1e6
    print(f'  OK  {BACKBONE_CKPT} ({sz:.0f} MB)')
else:
    print(f'  MISSING  {BACKBONE_CKPT}')
    print('  → Run Cell 1b to upload it.')

## Cell 5 — Drive helpers (save / restore)

In [ ]:
import shutil, os

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'

# attn4 is the new primary backbone (Run 57: OOD VDR 46.9%, best Stage-1 ever)
BACKBONE_RUN  = 'stage1_tcn_256_conformerattn4_lowlr'
BACKBONE_CKPT = f'{RUNS_DIR}/{BACKBONE_RUN}/checkpoints/best_metric.pth'

def restore_backbone():
    dst = f'/content/runs/{BACKBONE_RUN}/checkpoints/best_metric.pth'
    if os.path.exists(dst):
        print(f'Backbone already at {dst}')
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(BACKBONE_CKPT, dst)
    print(f'Restored backbone → {dst}')
    return dst

def restore_from_drive(run_name, ckpt_name='best_metric.pth'):
    src = f'{RUNS_DIR}/{run_name}/checkpoints/{ckpt_name}'
    dst_dir = f'/content/runs/{run_name}/checkpoints'
    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy2(src, f'{dst_dir}/{ckpt_name}')
    print(f'Restored {run_name}/{ckpt_name} ← Drive')
    return f'{dst_dir}/{ckpt_name}'

def save_to_drive(run_name):
    src = f'/content/runs/{run_name}'
    dst = f'{RUNS_DIR}/{run_name}'
    os.makedirs(RUNS_DIR, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Saved {run_name} → Drive')

backbone_ckpt = restore_backbone()
print(f'\nBackbone: {backbone_ckpt}')

---
## R1 — Deep probe from attn4 backbone (VocalSet + AnnotatedVocalSet only)

**Why:** Run 48 (deepprobe from attn2) got mF1 0.657 with OOD VDR 39.8%.  
Run 57 (attn4) has OOD VDR 46.9% — a stronger backbone should lift both.  
GTSinger tech labels are excluded: they hurt probe-mode mF1 (Run 52: 0.569 vs 0.657).

**Expected:** mF1 > 0.68, OOD VDR ≈ 46–48% (backbone frozen, VAD unchanged).

In [ ]:
!python vocalcoach/train.py \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 100 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode --deep-technique-head \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage2_attn4_deepprobe

In [ ]:
save_to_drive('stage2_attn4_deepprobe')

# OOD eval — backbone frozen so VDR should match attn4 backbone (46.9%)
!python scripts/evalOOD.py \
    --checkpoint /content/runs/stage2_attn4_deepprobe/checkpoints/best_metric.pth \
    --dataset vocadito \
    --data-dir /content/data/vocadito \
    --log results/ood_log.json \
    --regression-threshold 0.10

---
## R2 — Warm joint 20 epochs from attn4 backbone

**Why:** Run 54 (warmjoint from attn2) collapsed OOD VDR to 24.1% in just 20 epochs —  
but that used GTSinger tech (10k clips, strong technique gradient). Here we use  
VocalSet+AnnotatedVocalSet only (1.6k clips, much weaker gradient) at lr=1e-4 and w-technique=0.5.  
Run 55 (warmjoint→probe from attn2) achieved OOD VDR 44.0% despite the warm stage collapse —  
the probe step recovered it. From a stronger attn4 base, R2→R3 should push OOD VDR higher.

**Sequential: R3 depends on this checkpoint.**

In [ ]:
!python vocalcoach/train.py \
    --data-dir /content/data/merged_pitchvad \
    --noise-dir /content/data \
    --eval-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 20 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --w-vad 0.05 --w-pitch 2 --w-technique 0.5 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --lr 1e-4 --patience 20 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage2_attn4_warmjoint

In [ ]:
save_to_drive('stage2_attn4_warmjoint')

# --- R3: Deep probe from warm joint checkpoint ---
# Backbone now carries slight technique awareness from R2; probe head trains on top.
# Run 55 (attn2 version) got OOD VDR 44.0%, mF1 0.667. Expect improvement from attn4 base.
!python vocalcoach/train.py \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 80 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode --deep-technique-head \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30 \
    --resume /content/runs/stage2_attn4_warmjoint/checkpoints/best_metric.pth \
    --output-dir /content/runs/stage2_attn4_warmjoint_deepprobe

save_to_drive('stage2_attn4_warmjoint_deepprobe')

!python scripts/evalOOD.py \
    --checkpoint /content/runs/stage2_attn4_warmjoint_deepprobe/checkpoints/best_metric.pth \
    --dataset vocadito \
    --data-dir /content/data/vocadito \
    --log results/ood_log.json \
    --regression-threshold 0.10

---
## R4 — Joint difflr + deep technique head from attn4 backbone

**Why:** Run 53 (difflr from attn2) got mF1 0.676, OOD VDR 30.3% — best OOD-tested joint run.  
From the stronger attn4 base (OOD VDR 46.9%), the backbone has more headroom before collapse.  
`--lr-backbone 1e-5` keeps backbone shifts slow; deep head trains at 3e-4.  
VocalSet+AnnotatedVocalSet only — GTSinger tech excluded (dilutes gradient quality in probe; same logic applies here).

**Can run in parallel with R1.** OOD gate mandatory — accept only if OOD VDR > 35%.

In [ ]:
!python vocalcoach/train.py \
    --data-dir /content/data/merged_pitchvad \
    --noise-dir /content/data \
    --eval-dir /content/data \
    --technique-dirs /content/data/vocalset /content/data/annotated_vocalset \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 120 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --deep-technique-head \
    --lr 3e-4 --lr-backbone 1e-5 \
    --w-vad 0.05 --w-pitch 2 --w-technique 1 \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --patience 30 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage2_attn4_difflr

In [ ]:
save_to_drive('stage2_attn4_difflr')

!python scripts/evalOOD.py \
    --checkpoint /content/runs/stage2_attn4_difflr/checkpoints/best_metric.pth \
    --dataset vocadito \
    --data-dir /content/data/vocadito \
    --log results/ood_log.json \
    --regression-threshold 0.10

---
## Q1 — Quality V3 (SingMOS-Pro distillation) from R1

Probe-mode: backbone + technique head frozen. Only `head_quality` (scalar, 1-dim) trains.  
Phase 1 (ep 1–30): MSE on SingMOS-Pro pseudo-labels.  
Phase 2 (ep 31–60): ranking loss on PopBuTFy pairs.

**Run after R1 completes.** Evaluated with `evalQuality.py --mse-npz` (Spearman ρ).

In [ ]:
import os
r1_ckpt = '/content/runs/stage2_attn4_deepprobe/checkpoints/best_metric.pth'
if not os.path.exists(r1_ckpt):
    restore_from_drive('stage2_attn4_deepprobe')

!python vocalcoach/train.py \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 60 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode \
    --quality-variant 3 \
    --quality-mse-npz /content/data/quality/quality_mse.npz \
    --quality-pairs-npz /content/data/quality_50k/quality_pairs.npz \
    --quality-epochs-mse 30 \
    --w-quality-mse 1.0 --w-ranking 1.0 --ranking-margin 0.5 \
    --patience 20 \
    --resume {r1_ckpt} \
    --output-dir /content/runs/stage2_quality_v3_attn4

save_to_drive('stage2_quality_v3_attn4')

---
## Q2 — Quality V2 (9-dim ccmusic) from R1

9-dim output: Pitch / Rhythm / Timbre / Breath / Vibrato / Dynamic / Pronunciation / Vocal Range / Overall.  
Resumes from R1 directly (not from Q1 — incompatible head shapes: Q1 is 1-dim, Q2 needs 9-dim).  
Evaluated with `evalQuality.py --ccmusic-npz` (per-dim Spearman ρ).

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 60 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --probe-mode \
    --quality-variant 2 \
    --quality-mse-npz /content/data/quality/quality_mse.npz \
    --quality-ccmusic-npz /content/data/quality/quality_ccmusic.npz \
    --quality-pairs-npz /content/data/quality_50k/quality_pairs.npz \
    --quality-epochs-mse 30 \
    --w-quality-mse 1.0 --w-ranking 1.0 --ranking-margin 0.5 \
    --patience 20 \
    --resume {r1_ckpt} \
    --output-dir /content/runs/stage2_quality_v2_attn4

save_to_drive('stage2_quality_v2_attn4')

---
## EVAL — Quality head evaluation (Q1 and Q2)

Run `evalQuality.py` against both quality checkpoints.  
V3 → Spearman ρ vs SingMOS-Pro scores + ranking accuracy on PopBuTFy pairs.  
V2 → per-dim Spearman ρ vs ccmusic 9-dim expert labels.

In [ ]:
import os

# Q1 — V3 scalar (SingMOS-Pro distillation)
q1_ckpt = '/content/runs/stage2_quality_v3_attn4/checkpoints/best_metric.pth'
if not os.path.exists(q1_ckpt):
    restore_from_drive('stage2_quality_v3_attn4')

print('=== Q1 — V3 quality eval ===')
!python scripts/evalQuality.py \
    --checkpoint {q1_ckpt} \
    --mse-npz    /content/data/quality/quality_mse.npz \
    --pairs-npz  /content/data/quality_50k/quality_pairs.npz \
    --log results/quality_log.json

print()

# Q2 — V2 9-dim ccmusic
q2_ckpt = '/content/runs/stage2_quality_v2_attn4/checkpoints/best_metric.pth'
if not os.path.exists(q2_ckpt):
    restore_from_drive('stage2_quality_v2_attn4')

print('=== Q2 — V2 quality eval ===')
!python scripts/evalQuality.py \
    --checkpoint {q2_ckpt} \
    --ccmusic-npz /content/data/quality/quality_ccmusic.npz \
    --pairs-npz   /content/data/quality_50k/quality_pairs.npz \
    --log results/quality_log.json

---
## S1N — Stage-1 + note head from attn4 backbone

**Why note head must be Stage-1:** Run 56 proved that adding note onset/offset supervision  
to a Stage-2 checkpoint (even with a deep head) collapses OOD VDR to 23%. Note onset/offset  
detection is a pitch-domain task — it shares the same temporal edge-detection features as VAD.  
Training it jointly from Stage-1 lets all three heads (pitch, VAD, note) co-adapt, instead  
of retrofitting note gradients onto a backbone already optimised for pitch+VAD-only features.

**Uses `--deep-note-head`** and `--w-note 0.3` (lower than N1's 0.5) to protect pitch+VAD convergence.

#### S1N training

In [ ]:
!python vocalcoach/train.py \
    --data-dir /content/data/merged_pitchvad \
    --noise-dir /content/data \
    --eval-dir /content/data \
    --note-head --deep-note-head \
    --note-dirs /content/data/annotated_vocalset \
    --w-note 0.3 \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --epochs 150 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} \
    --w-vad 0.2 --w-pitch 2 \
    --vad-pos-weight 2.0 \
    --pitch-sigma 0.8 --augment noise_specaug \
    --lr 1e-3 --patience 40 \
    --resume {backbone_ckpt} \
    --output-dir /content/runs/stage1_attn4_notehead

In [ ]:
save_to_drive('stage1_attn4_notehead')

!python scripts/evalOOD.py \
    --checkpoint /content/runs/stage1_attn4_notehead/checkpoints/best_metric.pth \
    --dataset vocadito \
    --data-dir /content/data/vocadito \
    --log results/ood_log.json \
    --regression-threshold 0.10

---
## Download all checkpoints for local eval

In [ ]:
import os, zipfile
from google.colab import files

RUN_NAMES = [
    'stage2_attn4_deepprobe',
    'stage2_attn4_warmjoint_deepprobe',
    'stage2_attn4_difflr',
    'stage2_quality_v3_attn4',
    'stage2_quality_v2_attn4',
    'stage1_attn4_notehead',
]

zip_path = '/content/v2_checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for run in RUN_NAMES:
        for ckpt in ['best_metric.pth', 'best_loss.pth']:
            src = f'/content/runs/{run}/checkpoints/{ckpt}'
            if not os.path.exists(src):
                print(f'  SKIP: {run}/{ckpt}')
                continue
            arcname = f'vocalcoach/runs/{run}/checkpoints/{ckpt}'
            zf.write(src, arcname=arcname)
            print(f'  + {arcname}  ({os.path.getsize(src)/1e6:.1f} MB)')

print(f'\nZip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
files.download(zip_path)